# Старт в управлении: практическая часть

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.interpolate as scint
#all imports here pls

В этом задании мы детально изучим один из бриллиантов теории управления: линейный регулятор с квадратичным критерием (Linear Quadratic Regulator, LQR). Его стохастическая версия обычно называется LQG (Linear Quadratic Gaussian) из-за использования гауссовского шума (СДУ с Винеровским процессом в непрерывном случае и VAR(1) с управлением в дискретном случае).

Это один из примеров, когда 
1. И принцип максимума, и принцип динамического программирования приводят к одному результату примерно одинаковым усилием
2. Задача в непрерывном и в дискретном виде имеет практически идентичное решение с точностью до перевычисления констант
3. Задача имеет точное решение, которое является оптимальным в гауссовском случае и одновременно управлением с линейной обратной связью.

Про задачу LQR написано бесчисленное множество статей и обобщений. По сути в задачах управления это инструмент такого же уровня, как в машинном обучении -- градиентный бустинг или SVM.

Мы займёмся изучением задачи в непрерывном времени, потому что в дискретном происходит всё то же самое, но даже ещё проще.

Пусть $(\Omega, \mathcal{F}, P)$ вероятностное пространство с заданной фильтрацией $(\mathcal{F}_t)$ и поехали!

## Постановка задачи

Пусть $W$ -- Винеровский процесс, согласованный с фильтрацией $(\mathcal{F}_t)$. Рассматривается процесс состояния $X$, задающийся уравнением
$$
dX_t = (A X_t + B u_t + \mu) dt + \sigma dW_t, ~ X_0 = x_0 \in \mathbb{R}^d,
$$
где $A \in \mathbb{R}^{d \times d}, B \in \mathbb{R}^{d \times k}, \sigma \in \mathbb{R}^{d \times d}, \mu \in \mathbb{R}^d$. Требуется подобрать такое $u_t$ из общего класса управлений, чтобы достичь
$$
\min_u \mathbb{E}\left[ \int_0^T (X_t - x_*)^T Q (X_t - x_*) + u^T R u dt \right],
$$
где $x_* \in \mathbb{R}^d$ задано, $Q$ положительно полуопределена, а $R$ положительно определена, обе матрицы симметричны.

Формально это называется задачей отслеживания (LQR tracking).

1. $X_t$ -- процесс состояний (УСЛОВНО ориентация и высота самолёта, мы сейчас полагаем, что они измеряются точно).
2. $u_t$ -- управляющее воздействие (УСЛОВНО насколько прожата тяга в момент $t$).
3. $x_*$ -- целевое положение (где мы хотели бы видеть систему в идеале).
4. $x_0$ -- где система сейчас.

Можно подумать, что линейная динамика -- очень слабо, однако это не совсем так, когда дело доходит до реальных приложений, что демонстрирует например [Бернард Фридман в своей энциклопедии](https://store.doverpublications.com/products/9780486442785).

LQR как и фильтр Калмана -- простая и одновременно тяжело победимая во многих приложениях линейная модель.


## Вывод формул (5 баллов)

С помощью уравнения HJB получите формулы для
1. Оптимального управления $u_t=u(t,X_t)$ в форме с обратной связью.
2. Функции Беллмана $V_t(x)$.

В чём отличие от детерминированного случая?

ЗДЕСЬ МОГ БЫТЬ ВАШ КОММЕНТАРИЙ И РЕШЕНИЕ

## Простые обобщения (3 балла)


#### Терминант

Как поменяется вывод, если целевой критерий поменяется на
$$
\min_u \mathbb{E}\left[ \int_0^T \left\lbrace (X_t - x_*)^T Q (X_t - x_*) + u^T R u \right\rbrace dt ~+ ~(X_T-x_*)^T P (X_T-x_*) \right],
$$
где $P$ -- положительно полуопределённая симметричная матрица?


ЗДЕСЬ МОГ БЫТЬ ВАШ КОММЕНТАРИЙ И РЕШЕНИЕ

#### Зависимость $A,B, \sigma$ от времени

Как поменяется вывод, если матрицы $A,B,\sigma$ сделать зависимыми от времени?

ЗДЕСЬ МОГ БЫТЬ ВАШ КОММЕНТАРИЙ И РЕШЕНИЕ

## Имплементация и тестирование (17 баллов)

Для реализации удобнее всего реализовать класс `LQRController`, который будет выступать как сущность, обученная управлять системой, которую мы реализуем в классе `ControllableSDE`.


#### Управляемая среда

Начнём с управляемой среды, затем реализуем управление и в конце соберём это в единую сущность.

In [ ]:
class ControllableSDE:
    
    def __init__(self,A,B,sigma,mu=None,controller=None):
        """Controllable SDE of linear type

        dXt = (mu + AXt + But)dt + sigma dWt

        Args:
            A (float[]): dynamics matrix (d,d)
            B (float[]): control matrix (d,k)
            sigma (float[]): diffusion matrix (d,d)
            mu (float[], optional): fixed deviation (d,).
            controller (ControllerObj, optional): Controller instance implementing method computeControl(t,x) returning control u. If None, no control is used. Defaults to None.
        """        
        self.A = A
        self.B = B
        self.sigma = sigma
        if(mu is None):
            self.mu = np.zeros([self.A.shape[0]])
        else:
            self.mu = mu
        self.controller = controller
        
    def simulate(self,x0,ts):
        """Simulates set of trajectories

        Args:
            x0 (float[]): initial conditions (N,d)
            ts (float[]): time grid (T,)

        Returns:
            float[](N,d,T), float[](N,k,T-1) : states and controls
        """        
        #x0s N,d

        xs = np.zeros([x0.shape[0],x0.shape[1],ts.shape[0]]) #N d T
        us = np.zeros([x0.shape[0],self.B.shape[1],ts.shape[0]-1])
        xs[...,0] = x0
        ????????????????
        ??????????
            
        return xs, us
    
    def solverStep(self,t,x,dt):
        """Computes one solver step

        Args:
            t (float): time
            x (float[]): state
            dt (float): delta t

        Returns:
            x_new, u: new x and control/
        """        
        ????????????????
        ??????????    
        return x +??, u
    
    def _forwDEfun(self,t,x):
        """forwardDE function

        Args:
            t (float): time
            x (float[]): state (N,d) 

        Returns:
            float[](N,d): delta_state
        """
        # CAREFUL WITH STOCHASTIC PART TO GET THE RIGHT VARIANCE!        
        u = self.computeControl(t,x)
        return ????????????????
        ??????????
    
    def computeControl(self,t,x):
        """Computes control given by self.controller

        Args:
            t (dloat): time
            x (float[]): state (N,d)

        Returns:
            float[](N,k): controls
        """        
        if(self.controller is None):
            return np.zeros([x.shape[0],self.B.shape[1]])
        else:
            return self.controller.computeControl(t,x)

Проверим, что симуляция работает, попробуйте собрать следующий пример. Нарисуйте графики для переменных состояния в зависимости от времени.

Это процесс Орнштейна-Уленбека вида

$$
dX^i_t = (\mu^i - \theta^i X_t)dt + \sigma_{ii}dW^i_t, ~i=1,2
$$

для которого известно точное решение. Какое? Это вам поможет для теста.

ЗДЕСЬ МОГЛО БЫТЬ ВАШЕ РЕШЕНИЕ УРАВНЕНИЯ

In [ ]:
mu = np.array([0.5,0.2])
A = np.array([[-0.3,0],[0,-1]])
B = np.eye(2)
sigma = np.eye(2)*0.3
x0 = np.array([[1,-1],[3,-3]])
T=3
ts = np.linspace(0,T,200)

conSDE = ControllableSDE(A,B,sigma,mu)

In [ ]:
trajs, _ = conSDE.simulate(x0,ts)

In [ ]:
#Здесь мог быть ваш код для риования графиков
f, axs = plt.subplots(1,2,figsize=(10,4))

axs[0].set_title("Переменная X1",fontsize=16)
axs[0].set_xlabel("t",fontsize=14)
axs[0].grid()
axs[0].plot(ts,trajs[:,0,:].T)

axs[1].set_title("Переменная X2",fontsize=16)
axs[1].set_xlabel("t",fontsize=14)
axs[1].grid()
axs[1].plot(ts,trajs[:,1,:].T)


#### Управление

Теперь реализуем управление. По логике эта сущность должна передаваться в среду и внутри неё использоваться для генерации управляющих воздействий.

Для этого вам понадобится математический фундамент, который вы выводили выше, полезно выписать итоговые формулы здесь.

ЗДЕСЬ МОГЛИ БЫ БЫТЬ ВАШИ ФОРМУЛЫ ДЛЯ ВЫЧИСЛЕНИЯ УПРАВЛЕНИЯ

Будьте осторожны с численным решением ОДУ, так как матричные уравнения могут вести себя не очень устойчиво к ошибкам при использовании некоторых методов.

In [ ]:
    
class LQRController:
    
    def __init__(self, A,B,Q,R,mu):
        """LQR instance with xstar=const tracking

        Args:
            A (float[]): dynamics matrix (d,d)
            B (float[]): control matrix (d,k)
            Q (float[]): reconstruction error matrix from LQR (d,d)
            R (float[]): control cost matrix from LQR (k,k)
            mu (flaot[]): fixed deviation (d,)
        """        
        self.A = A
        self.B = B
        self.Q = Q
        self.R = R
        self.Rinv= np.linalg.inv(self.R)
        self.mu = mu
        self.initParams() #F,G,H=None
        
    def initParams(self):
        """
        Resets the parameters of LQR functions
        """        
        self.F = None #fun(t)
        self.G = None
        self.H = None
        self.fixedGain = None # numbers
        self.xstar = None
        
    def computeControl(self,t,x):
        """Computes LQR Control

        Args:
            t (float): time
            x (flaot[]): state (N,d)

        Returns:
           float[](N,k): controls
        """        
        u = ????????????????
        ??????????
        return u   
    
    def fit(self,xstar, ts, P=None):
        """Fits the LQR Bellman function parameters

        Args:
            xstar (float[]): steady-state point
            ts (float[]): time grid T,)
            P (float[], optional): Terminal condition for Riccati differential equation. Defaults to None.
        """        
        if(P is None):
            P = np.eye(self.Q.shape[0])
        self.fixedGain = 0.5*self.Rinv@(self.B.T)
        self.xstar = xstar
        
        rho = self.mu+ self.A@xstar
        
        Fs = np.zeros([len(ts),self.A.shape[0],self.A.shape[1]])
        Fs[0,...] = P
        Gs = np.zeros([len(ts),1,self.A.shape[1]])
        Hs = np.zeros([len(ts),1,1])
        BRBt = np.einsum("ij,jk,tk->it",self.B,self.Rinv,self.B)
        
        ????????????????
        ??????????
            
        #Fs = Fs[::-1,...]
        #Gs = Gs[::-1,...] #invert if you need
        #Hs = Hs[::-1,...]
        self.F = scint.make_interp_spline(ts, Fs, k=3) #!!!! important to have flexibility to change time grid if necessary
        self.G = scint.make_interp_spline(ts, Gs, k=3)
        self.H = scint.make_interp_spline(ts, Hs, k=3)
        
            
    def invDEfun(self,F0,G0,BRBt, rho):
        """Riccati and feedforward differential equation right part

        Args:
            F0 (float[]): quadratic form matrix (d,d)
            G0 (float[]): linear form matrix (1,d)
            BRBt (float[]): instrumental matrix  (d,d)
            rho (float[]): deviation (bias) (d,)

        Returns:
            dF,dG,dH: matrix deltas
        """        
        F = ????????????????
        ??????????
        G = ????????????????
        ??????????
        H = ????????????????
        ??????????
        
      
        return F,G,H

In [ ]:
mu = np.array([0.5,0.2])
A = np.array([[-0.3,0.2],[0,-1]])
B = np.eye(2)
Q = np.eye(2)*100
R = np.eye(2)*10
sigma = np.eye(2)*0.1
T=3
NT=200
ts = np.linspace(0,T,NT)
xstar=np.array([3,5]) #target point
NT=5000


contr = LQRController(A,B,Q,R,mu)
contr.fit(xstar,ts,P=-Q/10)
contrSDE = ControllableSDE(A,B,sigma,mu,controller=contr)


Нарисуйте графики состояний и управлений для данного примера.

In [ ]:
x0 = np.array([[1,-1],[3,-3]])
xs,us = contrSDE.simulate(x0,ts)

#Здесь мог быть ваш код для риования графиков
f, axs = plt.subplots(2,2,figsize=(10,10))

axs[0][0].grid()
axs[0][0].set_title("Переменная X1",fontsize=16)
axs[0][0].set_xlabel("t",fontsize=14)
axs[0][0].plot(ts,xs[:,0,:].T)

axs[0][1].grid()
axs[0][1].set_title("Переменная X2",fontsize=16)
axs[0][1].set_xlabel("t",fontsize=14)
axs[0][1].plot(ts,xs[:,1,:].T)

axs[1][0].grid()
axs[1][0].set_title("Управление U1",fontsize=16)
axs[1][0].set_xlabel("t",fontsize=14)
axs[1][0].plot(ts[:-1],us[:,0,:].T)

axs[1][1].grid()
axs[1][1].set_title("Управление U2",fontsize=16)
axs[1][1].set_xlabel("t",fontsize=14)
axs[1][1].plot(ts[:-1],us[:,1,:].T)

#plt.plot(ts,xs[0,:,:].T)
#plt.grid()
#plt.plot(ts[:-1],us[0,:,:].T)

Задача вроде решилась, но вы могли обратить внимание, что всё равно требуется настройка параметров. Как можно добиться максимального попадания в правый край и почему это может быть не так просто? Почему одно измерение может хорошо попадать в целевую точку $x_*$, а другое нет? Как добиться хорошего попадания? Как зависит результат управления от уровня шума $\sigma$?

Попробуйте разные гиперпараметры контроллера и попробуйте обосновать свои выводы.

ЗДЕСЬ МОГ БЫТЬ ВАШ КОММЕНТАРИЙ И ГРАФИКИ

In [ ]:
#больше графиков

#### Ещё простые обобщения (7 баллов, (всё, что выше в секции, за 10 баллов) )

Возможно, мы хотели бы отслеживать не одну точку $x_*$, а движущийся объект $x_*(t)$. Как мы увидим ниже, детерминированно движущийся объект -- это тоже взято из реальной задачи.

Как изменится вывод LQR, если $x_*(t)$ теперь является функцией от времени?

Напишите ниже обновлённые формулы.

ЗДЕСЬ МОГ БЫТЬ ВАШ ОТВЕТ И ОБОСНОВАНИЕ

Реализуйте `LQRTrackerController`, наследуя его от `LQRController`.

In [ ]:
class LQRTrackerController(LQRController):
    
    def fit(self,xstar, xstarDer, ts, P=None):
        """Fits the LQR Bellman function parameters for LQR tracker

        Args:
            xstar (func): steady-state function xstar(t)
            xstarDer (func): steady-state function derivative xstarDer(t)
            ts (float[]): time grid T,)
            P (float[], optional): Terminal condition for Riccati differential equation. Defaults to None.
        """       

        if(P is None):
            P = np.eye(self.Q.shape[0])
        self.fixedGain = 0.5*self.Rinv@(self.B.T)
        self.xstar = xstar
        self.xstarDer = xstarDer
        
        Fs = np.zeros([len(ts),self.A.shape[0],self.A.shape[1]])
        Fs[0,...] = P
        Gs = np.zeros([len(ts),1,self.A.shape[1]])
        Hs = np.zeros([len(ts),1,1])
        BRBt = np.einsum("ij,jk,tk->it",self.B,self.Rinv,self.B)
        ????????????????
        ??????????
            
        #Fs = Fs[::-1,...]
        #Gs = Gs[::-1,...]#invert if you need
        #Hs = Hs[::-1,...]
        self.F = scint.make_interp_spline(ts, Fs, k=3)
        self.G = scint.make_interp_spline(ts, Gs, k=3)
        self.H = scint.make_interp_spline(ts, Hs, k=3)
        
            
    def invDEfun(self,F0,G0,BRBt, rho):
        """Riccati and feedforward differential equation right part

        Args:
            F0 (float[]): quadratic form matrix (d,d)
            G0 (float[]): linear form matrix (1,d)
            BRBt (float[]): instrumental matrix  (d,d)
            rho (float[]): deviation (bias) (d,)

        Returns:
            dF,dG,dH: matrix deltas
        """        
        F = ????????????????
        ??????????
        G = ????????????????
        ??????????
        H = ????????????????
        ??????????
        
        
        return F,G,H
    
    def computeControl(self,t,x):
        """Computes LQR Tracker Control

        Args:
            t (float): time
            x (float[]): state (N,d)

        Returns:
            float[](N,k): controls
        """        
        
        u = ????????????????
        ??????????# n,d
        return u   

Зададим траекторию отслеживания, например, такую:

$$
x_*(t)=\begin{bmatrix}sin(t) \\ cos(2t)\end{bmatrix}
$$

Попробуйте построить для неё алгоритм отслеживания.

In [ ]:
mu = np.array([0.5,0.2])
A = np.array([[-0.3,0.2],[0,-1]])
B = np.eye(2)
Q = np.eye(2)*200
R = np.eye(2)*5
sigma = np.eye(2)*1
T=10
NT=500
ts = np.linspace(0,T,NT)
xstar=np.array([3,5]) #target point
NT=5000

def track(t):
    ????????????????
        ??????????
def trackDer(t):
    ????????????????
        ??????????

contr = LQRTrackerController(A,B,Q,R,mu)
contr.fit(track,trackDer,ts,P=-Q/10)
contrSDE = ControllableSDE(A,B,sigma,mu,controller=contr)

In [ ]:
x0 = np.array([[1,-1],[3,-3]])
xs,us = contrSDE.simulate(x0,ts)

#Здесь мог быть ваш код для риования графиков
f, axs = plt.subplots(2,2,figsize=(10,10))

axs[0][0].grid()
axs[0][0].set_title("Переменная X1",fontsize=16)
axs[0][0].set_xlabel("t",fontsize=14)
axs[0][0].plot(ts,xs[:,0,:].T)
axs[0][0].plot(ts,track(ts)[:,0],color="red")

axs[0][1].grid()
axs[0][1].set_title("Переменная X2",fontsize=16)
axs[0][1].set_xlabel("t",fontsize=14)
axs[0][1].plot(ts,xs[:,1,:].T)
axs[0][1].plot(ts,track(ts)[:,1],color="red")

axs[1][0].grid()
axs[1][0].set_title("Управление U1",fontsize=16)
axs[1][0].set_xlabel("t",fontsize=14)
axs[1][0].plot(ts[:-1],us[:,0,:].T)

axs[1][1].grid()
axs[1][1].set_title("Управление U2",fontsize=16)
axs[1][1].set_xlabel("t",fontsize=14)
axs[1][1].plot(ts[:-1],us[:,1,:].T)

#plt.plot(ts,xs[0,:,:].T)
#plt.grid()
#plt.plot(ts[:-1],us[0,:,:].T)

## Трекинг режима, определённого заранее (15 баллов)

В чём вообще смысл отслеживать детерминированную траекторию?

Например, вы хотите реализовать программу умного дома, которая должна поддерживать не самую горячую температуру днём, когда никого нет и чуть более холодную вечером и ночью для комфортного сна.

Тут в частности и появляется LQR и его потомки.

Пусть система климат-контроля задаётся двумя охлаждающими устройствами в разных комнатах:

$$
dX^i_t = r(x_e - X^i_t + u^i_t) dt + \sigma_{i\cdot} dW_t, ~~ r=0.15, ~ x_e=30,~T=24.
$$

в этот раз матрица $\sigma$ естественно полагается не диагональной, так как есть перемешивание воздуха между помещениями:

$$
\sigma = \begin{bmatrix}0.2 & -0.05\\ 0.02 & 0.3\end{bmatrix}.
$$

Целевая температура задаётся как

$$
x_*^1 = 27, x_*^2 = 29
$$

в периоде $[0,T/2)$ и 

$$
x_{**}^1 = 20, x_{**}^1 = 25
$$

в периоде $[T/2,T]$.

Постройте алгоритм управления в периоде $[0,T]$, предполагая, что стартовая температура может задаваться произвольно.

LQR -- достаточно мягкая задача, поэтому к трекингу нужно подойти с осторожностью, чтобы программа была выполнена как можно точнее. Можно сделать несколько контроллеров, по одному на каждый режим.

Постарайтесь добиться того, чтобы и днвной, и ночной режим обрабатывались достаточно точно и постройте графики температуры и управления.

In [ ]:
r=0.15
xe=30
mu = ??????
A = ??????
B = ??????
Q = np.eye(2)*15
R = np.eye(2)*1
sigma = ??????
T=24#hours
NT=1000
ts = np.linspace(0,T,NT)
xs1 = ?????? #regime1 steady state
xs2 = ?????? #regime2 steady state


def track(t):
    ??????
def trackDer(t):
    ??????
    
contr = LQRTrackerController(A,B,Q,R,mu)
contr.fit(track,trackDer,ts,P=-3*Q)
contrSDE = ControllableSDE(A,B,sigma,mu,controller=contr)

In [ ]:
x0 = np.array([[35,35]])
xs,us = contrSDE.simulate(x0,ts)

#Здесь мог быть ваш код для риования графиков
f, axs = plt.subplots(2,2,figsize=(10,10))

axs[0][0].grid()
axs[0][0].set_title("Температура X1",fontsize=16)
axs[0][0].set_xlabel("t",fontsize=14)
axs[0][0].plot(ts,xs[:,0,:].T)
axs[0][0].plot(ts,track(ts)[:,0],color="red")

axs[0][1].grid()
axs[0][1].set_title("Температура X2",fontsize=16)
axs[0][1].set_xlabel("t",fontsize=14)
axs[0][1].plot(ts,xs[:,1,:].T)
axs[0][1].plot(ts,track(ts)[:,1],color="red")

axs[1][0].grid()
axs[1][0].set_title("Управление U1",fontsize=16)
axs[1][0].set_xlabel("t",fontsize=14)
axs[1][0].plot(ts[:-1],us[:,0,:].T)

axs[1][1].grid()
axs[1][1].set_title("Управление U2",fontsize=16)
axs[1][1].set_xlabel("t",fontsize=14)
axs[1][1].plot(ts[:-1],us[:,1,:].T)


## Тяжёлый случай (10 баллов)

Давайте посмотрим, как регулятор справляется с более сложной динамикой. Рассмотрим CIR-процесс:

$$
dR_t = a(b-R_t)dt + \sigma \sqrt{R_t} dW_t, ~a>0,~b>0,
$$

Предположим, что выполнено условие Феллера $2ab \geq \sigma^2$ и положим, что мы достаточно далеко от нуля. Сможет ли его удержать линейный регулятор?

Нам потребуется новый класс `ControllableCIR`.

In [ ]:
class ControllableCIR:
    
    def __init__(self,a,b,sigma,controller=None):
        """Controllable CIR Process

        dR_t = a(b-R_t)dt + \sigma \sqrt{R_t} dW_t, ~a>0,~b>0,

        Args:
            a (float): a parameter
            b (float): b parameter
            sigma (float): volatility parameter
            controller (ControllerObj, optional): Controller implementing method computeControl(t,x). Defaults to None.
        """        
        self.a = a
        self.b = b
        self.sigma = sigma
        self.controller = controller
        
    def simulate(self,x0,ts):
        """Simulates set of trajectories

        Args:
            x0 (float[]): initial conditions (N,1)
            ts (float[]): time grid (T,)

        Returns:
            float[](N,1,T), float[](N,1,T-1) : states and controls
        """
        #x0s N,d
        xs = np.zeros([x0.shape[0],1,ts.shape[0]]) #N d T
        us = np.zeros([x0.shape[0],1,ts.shape[0]-1])
        xs[...,0] = x0
        ??????
            
        return xs, us
    
    def solverStep(self,t,x,dt):
        """Computes one solver step

        Args:
            t (float): time
            x (float[]): state (N,1)
            dt (float): delta t

        Returns:
            x_new, u: new x and control
        """        
        ??????      
        return x??????, u
    
    
    def _forwDEfun(self,t,x,dt):
        """forwardDE function

        Args:
            t (float): time
            x (float[]): state (N,1) 
            dt (float): delta time

        Returns:
            float[](N,1): delta_state
        """
        u = self.computeControl(t,x)
        return ??????, u
    
    def computeControl(self,t,x):
        """Computes control given by self.controller

        Args:
            t (dloat): time
            x (float[]): state (N,d)

        Returns:
            float[](N,k): controls
        """     
        
        if(self.controller is None):
            return np.zeros([x.shape[0],1])
        else:
            return self.controller.computeControl(t,x)

In [ ]:
a=3
b=2
sigma=0.2
cir = ControllableCIR(a,b,sigma)
T=10
NT=1000
ts = np.linspace(0,T,NT)
trajs,_ = cir.simulate(np.array([[5]]),ts)

In [ ]:
Alqr = np.array([[-a]])
Blqr= np.eye(1)
mu=np.array([a*b])
Q = np.eye(1)
R = np.eye(1)
xstar=np.array([2])
contr = LQRController(Alqr,Blqr,Q,R,mu)
contr.fit(xstar,ts,P=2*Q)

In [ ]:

#Здесь мог быть ваш код для риования графиков
f, axs = plt.subplots(2,1,figsize=(10,8))

axs[0].grid()
axs[0].set_title("CIR Rt",fontsize=16)
axs[0].set_xlabel("t",fontsize=14)
axs[0].plot(ts,trajs[:,0,:].T)

axs[1].grid()
axs[1].set_title("CIR Ut",fontsize=16)
axs[1].set_xlabel("t",fontsize=14)
axs[1].plot(ts[:-1],us[:,0,:].T)



Если CIR -- это модель ставки, то регулятор моделирует оптимальную интервенцию для стабилизации ставок на нужном уровне.

На самом деле из уравнения HJB легко вывести лучший регулятор, который учитывает зависимость волатильности от состояния. Но это задача на бонусный балл.

## Бонус (+5)

1. Решите задачу квадратичного регулятора для модели CIR, предполагая произвольный старт $R_0$ и заранее заданный целевой уровень $R_*$. Выведите формулы для управления и найдите функцию Беллмана.
2. Имплементируйте класс `CIRQRTrackerController` и сраните результат нового регулятора с результатом `LQRTrackerController`, построив график, как в примере выше.